In [8]:
from google.colab import files
uploaded = files.upload()

import pandas as pd
import plotly.express as px

# load data
df = pd.read_csv("merged_energy_emissions_clean.csv")

# keep only 2015 and 2023
df_change = df[df["year"].isin([2015, 2023])].copy()
df_change = df_change[["country", "year", "renewable_share"]].dropna()

# pivot data
pivot_df = df_change.pivot_table(
    index="country",
    columns="year",
    values="renewable_share",
    aggfunc="mean"
).reset_index()

# calculate change
pivot_df["renewable_change_pct_points"] = (pivot_df[2023] - pivot_df[2015]) * 100

# sort
change_df = pivot_df.sort_values("renewable_change_pct_points", ascending=False).copy()

# select biggest positive and negative changes
top_up = change_df.nlargest(10, "renewable_change_pct_points")
top_down = change_df.nsmallest(10, "renewable_change_pct_points")
final_df = pd.concat([top_up, top_down]).drop_duplicates()

# plot
fig = px.bar(
    final_df.sort_values("renewable_change_pct_points"),
    x="renewable_change_pct_points",
    y="country",
    orientation="h",
    text="renewable_change_pct_points",
    title="Biggest Increases and Decreases in Renewable Energy Share (2015 to 2023)"
)

fig.update_traces(
    texttemplate="%{text:.1f}",
    textposition="outside"
)

fig.update_layout(
    xaxis_title="Change in Renewable Energy (%)",
    yaxis_title="Country"
)

fig.show()

Saving merged_energy_emissions_clean.csv to merged_energy_emissions_clean (1).csv


In [10]:
# ==============================
# 1) INSTALL SCIENTIFIC COLORS
# ==============================
!pip install cmcrameri

# ==============================
# 2) IMPORT LIBRARIES
# ==============================
import pandas as pd
import numpy as np
import plotly.express as px
from cmcrameri import cm
from matplotlib.colors import to_hex

# ==============================
# 3) LOAD YOUR DATASET
# ==============================
df = pd.read_csv("merged_energy_emissions_clean.csv")

# ==============================
# 4) KEEP ONLY 2015 and 2023
# ==============================
renew = df[df["year"].isin([2015, 2023])].copy()

# ==============================
# 5) REMOVE DUPLICATE COUNTRIES
# ==============================
renew = renew.drop_duplicates(subset=["country", "year"])

# ==============================
# 6) CREATE PIVOT TABLE
# ==============================
pivot = renew.pivot(
    index="country",
    columns="year",
    values="renewable_share"
)

# remove countries missing either year
pivot = pivot.dropna()

# ==============================
# 7) CALCULATE CHANGE
# ==============================
pivot["renewable_change"] = pivot[2023] - pivot[2015]

# convert back to dataframe
final_df = pivot.reset_index()

# ==============================
# 8) TOP 10 INCREASE + TOP 10 DECREASE
# ==============================
top_positive = final_df.nlargest(10, "renewable_change")
top_negative = final_df.nsmallest(10, "renewable_change")

final_df = pd.concat([top_negative, top_positive])

# sort for better visual order
final_df = final_df.sort_values("renewable_change")

# ==============================
# 9) SCIENTIFIC DIVERGING COLOR
# ==============================
vik_colors = [to_hex(cm.vik(i)) for i in np.linspace(0, 1, 11)]
vik_scale = [[i/(len(vik_colors)-1), c] for i, c in enumerate(vik_colors)]

# ==============================
# 10) CREATE BEAUTIFUL CHART
# ==============================
fig = px.bar(
    final_df,
    x="renewable_change",
    y="country",
    color="renewable_change",
    orientation="h",
    text="renewable_change",
    title="🌍 Biggest Renewable Energy Share Changes (2015 → 2023)",
    color_continuous_scale=vik_scale
)

# bold labels
fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

# bold zero line
fig.add_vline(
    x=0,
    line_width=4,
    line_dash="solid",
    line_color="black"
)

# cleaner layout
fig.update_layout(
    template="plotly_white",
    title_font_size=24,
    xaxis_title="Change in Renewable Share (%)",
    yaxis_title="Country",
    height=800,
    width=1200,
    font=dict(size=14)
)

fig.show()

In [11]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# -----------------------------
# Load data
# -----------------------------
df = pd.read_csv("merged_energy_emissions_clean.csv")

# -----------------------------
# Keep only 2015 and 2023
# -----------------------------
renew = df[df["year"].isin([2015, 2023])].copy()

# keep only needed columns
renew = renew[["country", "year", "renewable_share"]].dropna()

# remove duplicate country-year rows if any
renew = renew.drop_duplicates(subset=["country", "year"])

# -----------------------------
# Pivot and calculate change
# -----------------------------
pivot = renew.pivot(
    index="country",
    columns="year",
    values="renewable_share"
)

# keep only countries that have both 2015 and 2023
pivot = pivot.dropna()

# percentage-point change
pivot["renewable_change"] = (pivot[2023] - pivot[2015]) * 100

# back to dataframe
change_df = pivot.reset_index()

# optional: keep biggest 10 increases and biggest 10 decreases
top_positive = change_df.nlargest(10, "renewable_change")
top_negative = change_df.nsmallest(10, "renewable_change")
final_df = pd.concat([top_negative, top_positive]).drop_duplicates()

# sort so negatives are first, positives later
final_df = final_df.sort_values("renewable_change")

# -----------------------------
# Green for positive, red for negative
# -----------------------------
final_df["bar_color"] = np.where(
    final_df["renewable_change"] >= 0,
    "#2ecc71",   # green
    "#e74c3c"    # red
)

# text labels
final_df["label"] = final_df["renewable_change"].round(1).astype(str) + "%"

# -----------------------------
# Plot
# -----------------------------
fig = go.Figure()

fig.add_trace(go.Bar(
    x=final_df["country"],
    y=final_df["renewable_change"],
    marker_color=final_df["bar_color"],
    text=final_df["label"],
    textposition="outside"
))

# bold zero line
fig.add_hline(
    y=0,
    line_width=3,
    line_color="black"
)

fig.update_layout(
    title="🌍 Biggest Renewable Energy Share Changes (2015 → 2023)",
    xaxis_title="Country",
    yaxis_title="Change in Renewable Energy Share (%)",
    template="plotly_white",
    height=700,
    width=1200,
    font=dict(size=14),
    showlegend=False
)

# make x labels easier to read
fig.update_xaxes(tickangle=45)

fig.show()

In [15]:
import pandas as pd
import plotly.express as px

# -----------------------------
# Load dataset
# -----------------------------
df = pd.read_csv("merged_energy_emissions_clean.csv")

# -----------------------------
# Keep 2015 and 2023
# -----------------------------
renew = df[df["year"].isin([2015, 2023])].copy()
renew = renew[["country", "year", "renewable_share"]].dropna()
renew = renew.drop_duplicates(subset=["country", "year"])

# -----------------------------
# Pivot and calculate change
# -----------------------------
pivot = renew.pivot(
    index="country",
    columns="year",
    values="renewable_share"
).dropna()

pivot["renewable_change"] = (pivot[2023] - pivot[2015]) * 100
final_df = pivot.reset_index()

# -----------------------------
# Top 10 positive + negative
# -----------------------------
top_positive = final_df.nlargest(10, "renewable_change")
top_negative = final_df.nsmallest(10, "renewable_change")

final_df = pd.concat([top_negative, top_positive]).drop_duplicates()
final_df = final_df.sort_values("renewable_change")

# -----------------------------
# Diverging red-green palette
# -----------------------------
import pandas as pd
import plotly.express as px

# -----------------------------
# Load dataset
# -----------------------------
df = pd.read_csv("merged_energy_emissions_clean.csv")

# -----------------------------
# Keep 2015 and 2023
# -----------------------------
renew = df[df["year"].isin([2015, 2023])].copy()
renew = renew[["country", "year", "renewable_share"]].dropna()
renew = renew.drop_duplicates(subset=["country", "year"])

# -----------------------------
# Pivot and calculate change
# -----------------------------
pivot = renew.pivot(
    index="country",
    columns="year",
    values="renewable_share"
).dropna()

pivot["renewable_change"] = (pivot[2023] - pivot[2015]) * 100
final_df = pivot.reset_index()

# -----------------------------
# Top 10 positive + negative
# -----------------------------
top_positive = final_df.nlargest(10, "renewable_change")
top_negative = final_df.nsmallest(10, "renewable_change")

final_df = pd.concat([top_negative, top_positive]).drop_duplicates()
final_df = final_df.sort_values("renewable_change")

# -----------------------------
# Plot
# -----------------------------
fig = px.bar(
    final_df,
    x="renewable_change",
    y="country",
    color="renewable_change",
    orientation="h",
    title="🌍 Biggest Renewable Energy Share Changes (2015 → 2023)",
    color_continuous_scale=[
        [0.0, "#c0392b"],
        [0.5, "#f5f5f5"],
        [1.0, "#27ae60"]
    ]
)

# remove bar labels
fig.update_traces(
    hovertemplate="<b>%{y}</b><br>Change: %{x:.1f}%<extra></extra>"
)

# bold zero line
fig.add_vline(
    x=0,
    line_width=4,
    line_color="black"
)

# wider layout
fig.update_layout(
    coloraxis_showscale=False,
    template="plotly_white",
    height=850,
    width=1600,   # wider
    margin=dict(l=260, r=140, t=80, b=80),
    title_font=dict(size=24),
    font=dict(size=15),
    xaxis_title="<b>Change in Renewable Share (%)</b>",
    yaxis_title="<b>Country</b>"
)

# nicer axis labels
fig.update_yaxes(
    tickfont=dict(size=15, color="#1f2937")
)

fig.update_xaxes(
    tickfont=dict(size=14),
    range=[
        final_df["renewable_change"].min() - 5,
        final_df["renewable_change"].max() + 8
    ]
)

fig.show()

In [57]:
import pandas as pd
import plotly.express as px

# -----------------------------
# Load dataset
# -----------------------------
df = pd.read_csv("merged_energy_emissions_clean.csv")

# -----------------------------
# Keep 2015 and 2023
# -----------------------------
renew = df[df["year"].isin([2015, 2023])].copy()
renew = renew[["country", "year", "renewable_share"]].dropna()
renew = renew.drop_duplicates(subset=["country", "year"])

# -----------------------------
# Pivot and calculate change
# -----------------------------
pivot = renew.pivot(
    index="country",
    columns="year",
    values="renewable_share"
).dropna()

pivot["renewable_change"] = (pivot[2023] - pivot[2015]) * 100
final_df = pivot.reset_index()

# -----------------------------
# Top 10 positive + negative
# -----------------------------
top_positive = final_df.nlargest(10, "renewable_change")
top_negative = final_df.nsmallest(10, "renewable_change")

final_df = pd.concat([top_negative, top_positive]).drop_duplicates()
final_df = final_df.sort_values("renewable_change")

# symmetric color scaling
max_abs = max(
    abs(final_df["renewable_change"].min()),
    abs(final_df["renewable_change"].max())
)

# -----------------------------
# Plot
# -----------------------------
fig = px.bar(
    final_df,
    x="renewable_change",
    y="country",
    color="renewable_change",
    orientation="h",
    text="renewable_change",
    title="🌍 Biggest Renewable Energy Share Changes (2015 → 2023)",
  color_continuous_scale="PiYG_r",
    range_color=[-200, 200]
)

# black outline for visibility
fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside",
    marker_line_color="black",
    marker_line_width=1.5
)

# bold zero line
fig.add_vline(
    x=0,
    line_width=4,
    line_color="black"
)

# layout
fig.update_layout(
    coloraxis_showscale=False,
    template="plotly_white",
    height=850,
    width=1600,
    margin=dict(l=260, r=140, t=80, b=80),
    title_font=dict(size=24),
    font=dict(size=15),
    xaxis_title="<b>Change in Renewable Share (%)</b>",
    yaxis_title="<b>Country</b>"
)

fig.update_yaxes(
    tickfont=dict(size=15, color="#1f2937")
)

fig.update_xaxes(
    tickfont=dict(size=14),
    range=[
        final_df["renewable_change"].min() - 5,
        final_df["renewable_change"].max() + 8
    ]
)

fig.update_xaxes(range=[-80,200])

fig.show()

In [31]:
import pandas as pd
import plotly.express as px

# -----------------------------
# Load dataset
# -----------------------------
df = pd.read_csv("merged_energy_emissions_clean.csv")

# -----------------------------
# Keep 2015 and 2023
# -----------------------------
co2 = df[df["year"].isin([2015, 2023])].copy()
co2 = co2[["country", "year", "co2_total"]].dropna()
co2 = co2.drop_duplicates(subset=["country", "year"])

# -----------------------------
# Pivot and calculate change
# -----------------------------
pivot = co2.pivot(
    index="country",
    columns="year",
    values="co2_total"
).dropna()

# percentage change from 2015 to 2023
pivot["co2_change"] = ((pivot[2023] - pivot[2015]) / pivot[2015]) * 100

final_df = pivot.reset_index()

# -----------------------------
# Top 10 increase + top 10 decrease
# -----------------------------
top_positive = final_df.nlargest(10, "co2_change")
top_negative = final_df.nsmallest(10, "co2_change")

final_df = pd.concat([top_negative, top_positive]).drop_duplicates()
final_df = final_df.sort_values("co2_change")

# -----------------------------
# Plot
# -----------------------------
fig = px.bar(
    final_df,
    x="co2_change",
    y="country",
    color="co2_change",
    orientation="h",
    text="co2_change",
    title="🌍 Biggest Carbon Emission Changes (2015 → 2023)",
    color_continuous_scale=[
        [0.0, "#27ae60"],   # green for decrease
        [0.5, "#f5f5f5"],   # white center
        [1.0, "#c0392b"]    # red for increase
    ]
)

# keep labels
fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

# bold zero line
fig.add_vline(
    x=0,
    line_width=4,
    line_color="black"
)

# layout
fig.update_layout(
    coloraxis_showscale=False,
    template="plotly_white",
    height=850,
    width=1600,
    margin=dict(l=260, r=140, t=80, b=80),
    title_font=dict(size=24),
    font=dict(size=15),
    xaxis_title="<b>Change in Carbon Emissions (%)</b>",
    yaxis_title="<b>Country</b>"
)

fig.update_yaxes(
    tickfont=dict(size=15, color="#1f2937")
)

fig.update_xaxes(
    tickfont=dict(size=14),
    range=[
        final_df["co2_change"].min() - 5,
        final_df["co2_change"].max() + 8
    ]
)

fig.show()

In [60]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("merged_energy_emissions_clean.csv")

co2 = df[df["year"].isin([2015, 2023])].copy()
co2 = co2[["country", "year", "co2_total"]].dropna()
co2 = co2.drop_duplicates(subset=["country", "year"])

pivot = co2.pivot(
    index="country",
    columns="year",
    values="co2_total"
).dropna()

# ✅ percentage change (same logic as renewable)
pivot["co2_change"] = ((pivot[2023] - pivot[2015]) / pivot[2015]) * 100

final_df = pivot.reset_index()

top_positive = final_df.nlargest(10, "co2_change")
top_negative = final_df.nsmallest(10, "co2_change")

final_df = pd.concat([top_negative, top_positive]).drop_duplicates()
final_df = final_df.sort_values("co2_change")

# symmetric colors
max_abs = max(
    abs(final_df["co2_change"].min()),
    abs(final_df["co2_change"].max())
)

fig = px.bar(
    final_df,
    x="co2_change",
    y="country",
    color="co2_change",
    orientation="h",
    text="co2_change",
    title="🌍 Biggest Carbon Emission Percentage Changes (2015 → 2023)",
    color_continuous_scale="PiYG_r",
    range_color=[-200, 200]
)

# black outlines
fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside",
    marker_line_color="black",
    marker_line_width=1.5
)

# bold zero divider
fig.add_vline(
    x=0,
    line_width=4,
    line_color="black"
)

fig.update_layout(
    coloraxis_showscale=False,
    template="plotly_white",
    height=850,
    width=1600,
    margin=dict(l=260, r=140, t=80, b=80),
    title_font=dict(size=24),
    font=dict(size=15),
    xaxis_title="<b>Change in Carbon Emissions (%)</b>",
    yaxis_title="<b>Country</b>"
)
fig.update_xaxes(range=[-80,200])

fig.show()

In [58]:
!pip install -U kaleido pycountry

import pandas as pd
import plotly.graph_objects as go
import pycountry
from google.colab import files

# -----------------------------
# Load dataset
# -----------------------------
df = pd.read_csv("merged_energy_emissions_clean.csv")

# -----------------------------
# Keep only 2023
# -----------------------------
latest = df[df["year"] == 2023].copy()

# -----------------------------
# Remove non-country / aggregate codes
# -----------------------------
BAD_CODES = {
    "CPNO", "NOEC", "ASOC", "OENA", "OEEU",
    "WP15", "WP16", "WP23", "WP27",
    "EURA", "OPEC"
}
latest = latest[~latest["code"].isin(BAD_CODES)].copy()

# -----------------------------
# Convert country names to ISO3
# -----------------------------
manual_iso3 = {
    "Russia": "RUS",
    "South Korea": "KOR",
    "North Korea": "PRK",
    "Iran": "IRN",
    "Turkey": "TUR",
    "Vietnam": "VNM",
    "United States": "USA",
    "United Kingdom": "GBR"
}

def get_iso3(country_name):
    if country_name in manual_iso3:
        return manual_iso3[country_name]
    try:
        return pycountry.countries.lookup(country_name).alpha_3
    except:
        return None

latest["iso3"] = latest["country"].apply(get_iso3)
latest = latest.dropna(subset=["iso3"]).copy()

# Remove duplicates if any remain
latest = latest.sort_values("co2_total", ascending=False)
latest = latest.drop_duplicates(subset=["country"])

# -----------------------------
# Top 10 emitters
# -----------------------------
top10 = latest.nlargest(10, "co2_total").copy()
top10["rank"] = range(1, len(top10) + 1)

# -----------------------------
# Approximate label positions
# (adjust if needed)
# -----------------------------
label_pos = {
    "China": {"lat": 35, "lon": 118, "text_lon": 135, "text_lat": 37},
    "United States": {"lat": 39, "lon": -98, "text_lon": -125, "text_lat": 40},
    "India": {"lat": 22, "lon": 78, "text_lon": 92, "text_lat": 10},
    "Russia": {"lat": 60, "lon": 90, "text_lon": 120, "text_lat": 62},
    "Japan": {"lat": 36, "lon": 138, "text_lon": 154, "text_lat": 34},
    "Iran": {"lat": 32, "lon": 53, "text_lon": 58, "text_lat": 25},
    "Germany": {"lat": 51, "lon": 10, "text_lon": 0, "text_lat": 48},
    "Saudi Arabia": {"lat": 24, "lon": 45, "text_lon": 42, "text_lat": 18},
    "Indonesia": {"lat": -2, "lon": 118, "text_lon": 130, "text_lat": -7},
    "South Korea": {"lat": 36, "lon": 128, "text_lon": 140, "text_lat": 20},
}

top10["lat"] = top10["country"].map(lambda x: label_pos[x]["lat"])
top10["lon"] = top10["country"].map(lambda x: label_pos[x]["lon"])
top10["text_lat"] = top10["country"].map(lambda x: label_pos[x]["text_lat"])
top10["text_lon"] = top10["country"].map(lambda x: label_pos[x]["text_lon"])

# -----------------------------
# Build map
# -----------------------------
fig = go.Figure()

# Background land/ocean
fig.update_geos(
    projection_type="natural earth",
    showland=True,
    landcolor="#7fb800",
    showocean=True,
    oceancolor="#bfe3f5",
    showcountries=True,
    countrycolor="white",
    coastlinecolor="white",
    showframe=False
)

# Dark country bubbles / anchor points
fig.add_trace(go.Scattergeo(
    lon=top10["lon"],
    lat=top10["lat"],
    mode="markers",
    marker=dict(
        size=18,
        color="black",
        line=dict(color="white", width=1)
    ),
    hoverinfo="skip",
    showlegend=False
))

# Thin connector lines
for _, row in top10.iterrows():
    fig.add_trace(go.Scattergeo(
        lon=[row["lon"], row["text_lon"]],
        lat=[row["lat"], row["text_lat"]],
        mode="lines",
        line=dict(color="white", width=1.5),
        hoverinfo="skip",
        showlegend=False
    ))

# Label boxes using annotations
for _, row in top10.iterrows():
    label = (
        f"<b>{int(row['rank'])}. {row['country']}</b><br>"
        f"{row['co2_total']:,.0f} MtCO₂"
    )
    fig.add_annotation(
        x=row["text_lon"],
        y=row["text_lat"],
        xref="x",
        yref="y",
        text=label,
        showarrow=False,
        font=dict(size=12, color="black"),
        bgcolor="white",
        bordercolor="#999999",
        borderwidth=1,
        borderpad=4
    )

# Because geo plots do not use x/y axes normally, add invisible scatter for annotations
fig.add_trace(go.Scattergeo(
    lon=top10["text_lon"],
    lat=top10["text_lat"],
    mode="markers",
    marker=dict(size=1, color="rgba(0,0,0,0)"),
    hoverinfo="skip",
    showlegend=False
))

fig.update_layout(
    title=dict(
        text="<b>Countries With the Highest Carbon Footprint</b><br><sup>Top 10 CO₂ emitters in 2023 (MtCO₂)</sup>",
        x=0.02,
        xanchor="left",
        font=dict(size=24)
    ),
    height=750,
    width=1300,
    margin=dict(l=10, r=10, t=90, b=10),
    paper_bgcolor="white"
)

fig.show()